In [2]:
import os
import numpy as np
from PIL import Image
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# ==============================
# DATASET PATH
# ==============================
DATA_PATH = r"C:\Users\Supreeth TS\Downloads\leapGestRecog\leapGestRecog"

IMG_SIZE = 64

# ==============================
# LOAD DATA FUNCTION
# ==============================
def load_dataset(path):
    X_data = []
    y_data = []

    print("Loading dataset...")

    for person in os.listdir(path):
        person_path = os.path.join(path, person)

        if not os.path.isdir(person_path):
            continue

        for gesture in os.listdir(person_path):
            gesture_path = os.path.join(person_path, gesture)

            if not os.path.isdir(gesture_path):
                continue

            for img_name in os.listdir(gesture_path):
                img_path = os.path.join(gesture_path, img_name)

                try:
                    img = Image.open(img_path).convert('L')
                    img = img.resize((IMG_SIZE, IMG_SIZE))
                    img = np.array(img)

                    features = hog(
                        img,
                        orientations=9,
                        pixels_per_cell=(8, 8),
                        cells_per_block=(2, 2),
                        block_norm='L2-Hys'
                    )

                    X_data.append(features)
                    y_data.append(gesture)

                except Exception as e:
                    continue

    return np.array(X_data), np.array(y_data)


# ==============================
# LOAD DATA
# ==============================
X, y = load_dataset(DATA_PATH)

print("Dataset loaded:", X.shape)

# ==============================
# LABEL ENCODING (IMPORTANT FIX)
# ==============================
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# ==============================
# TRAIN TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

# ==============================
# PIPELINE (SCALER + SVM)
# ==============================
model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=5, gamma='scale'))
])

print("Training model...")
model.fit(X_train, y_train)

# ==============================
# PREDICTION
# ==============================
y_pred = model.predict(X_test)

# ==============================
# EVALUATION
# ==============================
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ==============================
# SAVE MODEL
# ==============================
joblib.dump({
    "model": model,
    "label_encoder": le
}, "hand_gesture_model.pkl")

print("\nModel saved successfully!")

Loading dataset...
Dataset loaded: (20000, 1764)
Training model...

Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00       400
           2       1.00      1.00      1.00       400
           3       1.00      1.00      1.00       400
           4       1.00      1.00      1.00       400
           5       1.00      1.00      1.00       400
           6       1.00      1.00      1.00       400
           7       1.00      1.00      1.00       400
           8       1.00      1.00      1.00       400
           9       1.00      1.00      1.00       400

    accuracy                           1.00      4000
   macro avg       1.00      1.00      1.00      4000
weighted avg       1.00      1.00      1.00      4000


Confusion Matrix:
 [[400   0   0   0   0   0   0   0   0   0]
 [  0 400   0   0   0   0   0   0   0   0]
 [  0   0 400   0   0   0   